In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import date, timedelta
from pandas.tseries.offsets import *
import calendar

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

year = 2026
quarter = 1
today = date.today()
yesterday = today - timedelta(days=1)
print(today, yesterday)

2026-05-22 2026-05-21


In [3]:
num_business_days = BDay(1)
yesterday = today - num_business_days
yesterday = yesterday.date()
print(today, yesterday)

2026-05-22 2026-05-21


### Today must be last business day

In [63]:
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(1)
yesterday = today - num_business_days
print(f'today: {today}')
to_mmdd = today.strftime('%m-%d')
yesterday = yesterday.date()
print(f'Today in mm-dd format: {to_mmdd}')
print(f'yesterday: {yesterday}')

today: 2026-05-20
Today in mm-dd format: 05-20
yesterday: 2026-05-19


In [65]:
format_dict = {    
    'shares':'{:,}',    
    'q4':'{:.4f}','q3':'{:.4f}','q2':'{:.4f}','q1':'{:.4f}','dividend':'{:.4f}','qtrly':'{:.4f}',
    'price':'{:.2f}',    
    'amount':'{:,.2f}','net':'{:,.2f}','cost_amt':'{:,.2f}',
    'yield':'{:,.2f}%','pct':'{:,.2f}%',
    'xdate':'{:%Y-%m-%d}','paiddate':'{:%Y-%m-%d}',
              }

In [67]:
# Get the user's home directory
user_path = os.path.expanduser('~')
# Get the current working directory
current_path = os.getcwd()
# Derive the base directory (base_dir) by removing the last folder ('Daily')
base_path = os.path.dirname(current_path)
#C:\Users\PC1\OneDrive\A5\Data
dat_path = os.path.join(base_path, "Data")
#C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data>
god_path = os.path.join(user_path, "OneDrive","Imports","santisoontarinka@gmail.com - Google Drive","Data")
#C:\Users\PC1\iCloudDrive\data
icd_path = os.path.join(user_path, "iCloudDrive", "Data")
#C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data
osd_path = os.path.join(user_path, "OneDrive","Documents","obsidian-git-sync","Data")
#C:\Users\PC1\OneDrive\A5\Excel
xsl_path = os.path.join(base_path, "Excel")

In [69]:
print("User path:", user_path)
print(f"Current path: {current_path}")
print(f"Base path: {base_path}")
print(f"Data path (dat_path): {dat_path}") 
print(f"Excel path (xsl_path): {xsl_path}") 
print(f"Google Drive path (god_path): {god_path}")
print(f"iCloudDrive path (icd_path): {icd_path}") 
print(f"Obsidian path (osd_path): {osd_path}")

User path: C:\Users\PC1
Current path: C:\Users\PC1\OneDrive\A4\Ad hoc
Base path: C:\Users\PC1\OneDrive\A4
Data path (dat_path): C:\Users\PC1\OneDrive\A4\Data
Excel path (xsl_path): C:\Users\PC1\OneDrive\A4\Excel
Google Drive path (god_path): C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data
iCloudDrive path (icd_path): C:\Users\PC1\iCloudDrive\Data
Obsidian path (osd_path): C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data


### New dividend stock

In [4]:
name = 'WHAIR'
sql = """
SELECT * 
FROM DIVIDEND
WHERE name = '%s'
"""
sql = sql % name
dividend = pd.read_sql(sql, const)
dividend.drop(['PRICE', 'PERCENT'], axis=1, inplace=True)
dividend.columns = dividend.columns.str.lower()
dividend['shares'] = dividend['shares'].astype('int64')
dividend['xdate'] = pd.to_datetime(dividend['xdate'])
dividend['paiddate'] = pd.to_datetime(dividend['paiddate'])
dividend.style.format(format_dict)

,name,q4,q3,q2,q1,dividend,shares,xdate,paiddate,kind,actual


In [3]:
sqlIns = """
INSERT INTO dividend
VALUES('xxx',.1738,0.1738,0.1738,0.1556,0,0,0,00,'2022-05-25','2022-06-22','',0)
"""
rp = const.execute(sqlIns)
rp.rowcount

ObjectNotExecutableError: Not an executable object: "\nINSERT INTO dividend\nVALUES('xxx',.1738,0.1738,0.1738,0.1556,0,0,0,00,'2022-05-25','2022-06-22','',0)\n"

### Start of Update dividend

In [71]:
def update_dividend1(q1, XDATE, PAIDDATE, actual, name):
    # Use parameterized query to avoid SQL injection
    sql = text("""
        UPDATE dividend 
        SET q1 = :q1, 
            dividend = q1 + q2 + q3 + q4, 
            XDATE = :XDATE, 
            PAIDDATE = :PAIDDATE, 
            actual = :actual 
        WHERE name = :name
    """)
    
    # Execute the query with parameters
    rp = const.execute(sql, {
        'q1': q1,
        'XDATE': XDATE,
        'PAIDDATE': PAIDDATE,
        'actual': actual,
        'name': name
    })
    const.commit()
    return f"Records updated = {rp.rowcount}"

In [101]:
name = 'TFFIF'
sql = """
SELECT * FROM DIVIDEND WHERE name = '%s'"""
sql = sql % name
dividend = pd.read_sql(sql, const)
dividend.drop(['PRICE', 'PERCENT'], axis=1, inplace=True)
dividend.columns = dividend.columns.str.lower()
dividend['shares'] = dividend['shares'].astype('int64')
dividend['xdate'] = pd.to_datetime(dividend['xdate'])
dividend['paiddate'] = pd.to_datetime(dividend['paiddate'])
dividend.style.format(format_dict)

,name,q4,q3,q2,q1,dividend,shares,xdate,paiddate,kind,actual
0,TFFIF,0.1140,0.1128,0.1176,0.1219,0.4663,"25,000",2026-02-27,2026-03-18,,0


In [103]:
q1 = 0.1204
XDATE = '2026-05-27'
PAIDDATE = '2026-06-16'
actual = 1

update_dividend1(q1,XDATE,PAIDDATE,actual,name)

'Records updated = 1'

In [105]:
sql = """
SELECT * FROM DIVIDEND WHERE name = '%s'"""
sql = sql % name
dividend = pd.read_sql(sql, const)
dividend.drop(['PRICE', 'PERCENT'], axis=1, inplace=True)
dividend.columns = dividend.columns.str.lower()
dividend['shares'] = dividend['shares'].astype('int64')
dividend['xdate'] = pd.to_datetime(dividend['xdate'])
dividend['paiddate'] = pd.to_datetime(dividend['paiddate'])
dividend.style.format(format_dict)

,name,q4,q3,q2,q1,dividend,shares,xdate,paiddate,kind,actual
0,TFFIF,0.1140,0.1128,0.1176,0.1204,0.4648,"25,000",2026-05-27,2026-06-16,,1


In [99]:
def update_dividend2(shares, q1, actual, name):
    # Use parameterized query to avoid SQL injection
    sql = text("""
        UPDATE dividend 
        SET q1 = :q1, 
            dividend = q1 + q2 + q3 + q4, 
            shares = :shares, 
            actual = :actual 
        WHERE name = :name
    """)
    
    # Execute the query with parameters
    rp = const.execute(sql, {
        'q1': q1,
        'shares': shares,
        'actual': actual,
        'name': name
    })
    
    return f"Records updated = {rp.rowcount}"

In [31]:
sql = """
SELECT * 
FROM DIVIDEND
WHERE name = '%s'
"""
sql = sql % name
dividend = pd.read_sql(sql, const)
dividend.drop(['PRICE', 'PERCENT'], axis=1, inplace=True)
dividend.columns = dividend.columns.str.lower()
dividend['shares'] = dividend['shares'].astype('int64')
dividend['xdate'] = pd.to_datetime(dividend['xdate'])
dividend['paiddate'] = pd.to_datetime(dividend['paiddate'])
dividend.style.format(format_dict)

,name,q4,q3,q2,q1,dividend,shares,xdate,paiddate,kind,actual
0,GVREIT,0.2044,0.1911,0.1963,0.2050,0.7968,"60,000",2025-02-26,2025-03-12,,1


In [9]:
shares = 20000
q1 = 0
actual = 0

In [11]:
update_dividend2(shares,q1,actual,name)

'Records updated = 1'

### Toggle actual status

In [2]:
name = 'BCT'
actual = 0
sqlUpd = "UPDATE dividend SET actual = %s WHERE name = '%s'"
sqlUpd = sqlUpd % (actual, name)
rp = const.execute(sqlUpd)
rp.rowcount

1

### Delete dividend record

In [10]:
sqlDel = '''
DELETE FROM dividend
WHERE name IN ("MC")
'''
#rp = const.execute(sqlDel)
rp.rowcount

1

### Start of output to cloud

In [107]:
cols = 'name qtrly shares amount net xdate paiddate cost_amt pct actual'.split()

In [109]:
sql = """
SELECT Y.NAME AS name,  Q1 AS qtrly, SHARES, XDATE, PAIDDATE, 
P.price AS price, Y.DIVIDEND, ACTUAL, B.price * B.volbuy AS cost_amt
FROM dividend AS Y, price AS P, buy AS B
WHERE Y.name = P.name 
AND Y.name = B.name 
AND Q1 > 0 
AND P.date = '%s'
ORDER BY name
"""
sql = sql % yesterday 
print(sql)


SELECT Y.NAME AS name,  Q1 AS qtrly, SHARES, XDATE, PAIDDATE, 
P.price AS price, Y.DIVIDEND, ACTUAL, B.price * B.volbuy AS cost_amt
FROM dividend AS Y, price AS P, buy AS B
WHERE Y.name = P.name 
AND Y.name = B.name 
AND Q1 > 0 
AND P.date = '2026-05-19'
ORDER BY name



In [111]:
dividend = pd.read_sql(sql, const)
dividend.columns = dividend.columns.str.lower()
dividend['shares'] = dividend['shares'].astype('int64')
dividend['xdate'] = pd.to_datetime(dividend['xdate'])
dividend['paiddate'] = pd.to_datetime(dividend['paiddate'])
dividend['amount'] = round(dividend['shares'] * dividend['qtrly'], 2)
dividend['net'] = round(dividend['amount'] * 0.9, 2)
dividend['pct'] = round(dividend['net'] / dividend['cost_amt'] * 100, 2)
dividend[cols].style.format(format_dict)

,name,qtrly,shares,amount,net,xdate,paiddate,cost_amt,pct,actual
0,CPNREIT,0.2800,"55,000","15,400.00","13,860.00",2026-05-28,2026-06-11,"990,000.00",1.40%,1
1,DIF,0.2222,"45,000","9,999.00","8,999.10",2026-05-15,2026-06-10,"571,500.00",1.57%,1
2,GVREIT,0.1946,"69,000","13,427.40","12,084.66",2026-05-26,2026-06-11,"534,750.00",2.26%,1
3,IVL,0.1750,"7,200","1,260.00","1,134.00",2026-05-28,2026-06-12,"288,000.00",0.39%,1
4,TFFIF,0.1204,"25,000","3,010.00","2,709.00",2026-05-27,2026-06-16,"182,500.00",1.48%,1
5,WHAIR,0.1325,"50,000","6,625.00","5,962.50",2026-03-05,2026-03-27,"435,000.00",1.37%,0
6,WHART,0.1915,"20,000","3,830.00","3,447.00",2026-05-19,2026-06-05,"246,000.00",1.40%,1


In [113]:
dividend.net.sum()

48196.259999999995

In [115]:
file_name = 'dividend-q1.csv'
output_file = os.path.join(dat_path, file_name)
god_file = os.path.join(god_path, file_name)
icd_file = os.path.join(icd_path, file_name)
osd_file = os.path.join(osd_path, file_name)

In [117]:
print(f"Output file : {output_file}") 
print(f"icd_file : {icd_file}") 
print(f"god_file : {god_file}") 
print(f"osd_file : {osd_file}") 

Output file : C:\Users\PC1\OneDrive\A4\Data\dividend-q1.csv
icd_file : C:\Users\PC1\iCloudDrive\Data\dividend-q1.csv
god_file : C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data\dividend-q1.csv
osd_file : C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data\dividend-q1.csv


In [119]:
dividend[cols].sort_values(['name'],ascending=[True]).to_csv(output_file, index=False)
dividend[cols].sort_values(['name'],ascending=[True]).to_csv(god_file, index=False)
dividend[cols].sort_values(['name'],ascending=[True]).to_csv(osd_file, index=False)
dividend[cols].sort_values(['name'],ascending=[True]).to_csv(icd_file, index=False)

In [121]:
file_name = 'dividend-q1.xlsx'
data_file = dat_path + file_name
output_file = dat_path + file_name
osd_file = osd_path + file_name

dividend[cols].sort_values(['name'],ascending=[True]).to_excel(output_file, index=False)
dividend[cols].sort_values(['name'],ascending=[True]).to_excel(data_file, index=False)
dividend[cols].sort_values(['name'],ascending=[True]).to_excel(osd_file, index=False)

### End of output to cloud

In [123]:
sql = 'SELECT name, q_eps, aq_eps, publish_date FROM epss WHERE year = %s-1 AND quarter = %s ORDER BY name'
sql = sql % (year, quarter)
print(sql)
eps = pd.read_sql(sql, conlt)
eps.dtypes

SELECT name, q_eps, aq_eps, publish_date FROM epss WHERE year = 2026-1 AND quarter = 1 ORDER BY name


name             object
q_eps           float64
aq_eps          float64
publish_date     object
dtype: object

In [125]:
df_merge = pd.merge(dividend, eps, on='name', how='inner')
df_merge.sort_values('name',ascending=True)

,name,qtrly,shares,xdate,paiddate,price,dividend,actual,cost_amt,amount,net,pct,q_eps,aq_eps,publish_date
0,CPNREIT,0.2800,55000,2026-05-28,2026-06-11,12.6,0.8546,1,990000.0,15400.0,13860.00,1.40,0.00,0.00,2025-05-08
1,DIF,0.2222,45000,2026-05-15,2026-06-10,9.8,0.8888,1,571500.0,9999.0,8999.10,1.57,0.00,0.00,2025-05-14
2,GVREIT,0.1946,69000,2026-05-26,2026-06-11,6.9,0.7678,1,534750.0,13427.4,12084.66,2.26,0.00,0.00,2025-02-13
3,IVL,0.1750,7200,2026-05-28,2026-06-12,24.4,0.7000,1,288000.0,1260.0,1134.00,0.39,-0.27,-0.27,2025-05-09
4,TFFIF,0.1204,25000,2026-05-27,2026-06-16,6.6,0.4648,1,182500.0,3010.0,2709.00,1.48,0.00,0.00,2025-02-13
5,WHAIR,0.1325,50000,2026-03-05,2026-03-27,7.4,0.5730,0,435000.0,6625.0,5962.50,1.37,0.00,0.00,2025-05-14
6,WHART,0.1915,20000,2026-05-19,2026-06-05,10.4,0.6774,1,246000.0,3830.0,3447.00,1.40,0.00,0.00,2025-05-07


In [131]:
print(sql)

SELECT name, q_eps, aq_eps, publish_date FROM epss WHERE year = 2026-1 AND quarter = 1 ORDER BY name


In [133]:
sql = """
SELECT Y.NAME AS name,  Q1 AS qtrly, SHARES, XDATE, PAIDDATE, 
P.price AS price, Y.DIVIDEND, ACTUAL, B.price * B.volbuy AS cost_amt
FROM dividend AS Y, price AS P, buy AS B
WHERE Y.name = P.name 
AND Y.name = B.name 
AND Q1 > 0 
AND P.date = '%s'
ORDER BY name
"""
sql = sql % yesterday 
print(sql)


SELECT Y.NAME AS name,  Q1 AS qtrly, SHARES, XDATE, PAIDDATE, 
P.price AS price, Y.DIVIDEND, ACTUAL, B.price * B.volbuy AS cost_amt
FROM dividend AS Y, price AS P, buy AS B
WHERE Y.name = P.name 
AND Y.name = B.name 
AND Q1 > 0 
AND P.date = '2026-05-19'
ORDER BY name



In [140]:
dividend = pd.read_sql(sql, const)
dividend.columns = dividend.columns.str.lower()
dividend['shares'] = dividend['shares'].astype('int64')
dividend['xdate'] = pd.to_datetime(dividend['xdate'])
dividend['paiddate'] = pd.to_datetime(dividend['paiddate'])
dividend['amount'] = round(dividend['shares'] * dividend['qtrly'], 2)
#dividend['amt_yearly'] = round(dividend['shares'] * dividend['yearly'], 2)
dividend['net'] = round(dividend['amount'] * 0.9, 2)
dividend['pct'] = round(dividend['net'] / dividend['cost_amt'] * 100, 2)
#dividend['yearly'] = round(dividend['amt_yearly'] / dividend['cost_amt'] * 100, 2)
dividend[cols].style.format(format_dict)

,name,qtrly,shares,amount,net,xdate,paiddate,cost_amt,pct,actual
0,CPNREIT,0.2800,"55,000","15,400.00","13,860.00",2026-05-28,2026-06-11,"990,000.00",1.40%,1
1,DIF,0.2222,"45,000","9,999.00","8,999.10",2026-05-15,2026-06-10,"571,500.00",1.57%,1
2,GVREIT,0.1946,"69,000","13,427.40","12,084.66",2026-05-26,2026-06-11,"534,750.00",2.26%,1
3,IVL,0.1750,"7,200","1,260.00","1,134.00",2026-05-28,2026-06-12,"288,000.00",0.39%,1
4,TFFIF,0.1204,"25,000","3,010.00","2,709.00",2026-05-27,2026-06-16,"182,500.00",1.48%,1
5,WHAIR,0.1325,"50,000","6,625.00","5,962.50",2026-03-05,2026-03-27,"435,000.00",1.37%,0
6,WHART,0.1915,"20,000","3,830.00","3,447.00",2026-05-19,2026-06-05,"246,000.00",1.40%,1


In [146]:
colt = 'name qtrly shares amount net publish_date xdate paiddate'.split()

In [148]:
df_merge[colt].sort_values('publish_date',ascending=True)

,name,qtrly,shares,amount,net,publish_date,xdate,paiddate
2,GVREIT,0.1946,69000,13427.4,12084.66,2025-02-13,2026-05-26,2026-06-11
4,TFFIF,0.1204,25000,3010.0,2709.00,2025-02-13,2026-05-27,2026-06-16
6,WHART,0.1915,20000,3830.0,3447.00,2025-05-07,2026-05-19,2026-06-05
0,CPNREIT,0.2800,55000,15400.0,13860.00,2025-05-08,2026-05-28,2026-06-11
3,IVL,0.1750,7200,1260.0,1134.00,2025-05-09,2026-05-28,2026-06-12
1,DIF,0.2222,45000,9999.0,8999.10,2025-05-14,2026-05-15,2026-06-10
5,WHAIR,0.1325,50000,6625.0,5962.50,2025-05-14,2026-03-05,2026-03-27


In [150]:
file_name = '24-div-26q1.xlsx'
xsl_file = os.path.join(xsl_path, file_name)

df_merge[colt].sort_values(['name'],ascending=[True]).to_excel(xsl_file, index=False)

In [54]:
#const.commit()
#const.close()